### Loading

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

df= pd.read_csv("../data/processed/upl_goals_2019_2025_cleaned.csv")

In [2]:
# df.head()
# df.shape
# df.info()
# df.describe()
# df.isna().sum()

### Preprocessing

In [3]:
# normalize column names: strip whitespace, lowercase, replace spaces (and other whitespace) with underscores
df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)

In [4]:
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d', errors='coerce') # parse ISO-format dates (YYYY-MM-DD); coerce invalid formats to NaT
df['goal_minute_num'] = pd.to_numeric(df['goal_minute_num'], errors='coerce') 

In [5]:
# strip leading/trailing whitespace for all object dtype columns in-place
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].apply(lambda v: v.strip() if isinstance(v, str) else v)

In [6]:
df.replace("", np.nan, inplace=True)

In [7]:
df = df[df['season'] != '2025/26'] # exclude 2025/26 season which is ongoing and has incomplete data

### Regular Time Goals

In [8]:
dfr = df[df['in_added_time']== 0] #excluse goals which were scored in added/extra time

### Added Time Goals

In [9]:
dfa = df[df['in_added_time']== 1] #include goals which were scored in added/extra time

## Other Derived Metrics

### Creating Scoreline after each goal

In [10]:
# --- DATA CLEANING (CRITICAL) ---
# Ensure we can match 'Own Goal' regardless of capitalization (e.g., 'own goal', 'Own Goal', 'OG')
# Adjust 'Own Goal' string based on exactly what is in your scraped data
df['goal_type_clean'] = df['goal_type'].astype(str).str.lower().str.strip()
df['team_side_clean'] = df['team_side'].astype(str).str.lower().str.strip() 

# --- DEFINING THE CONDITIONS ---

# Condition 1: When does the HOME TEAM get a point?
# Case A: Home Team scores AND it is NOT an own goal
# Case B: Away Team scores AND it IS an own goal
home_point_conditions = [
    (df['team_side_clean'] == 'home') & (df['goal_type_clean'] != 'own goal'),
    (df['team_side_clean'] == 'away') & (df['goal_type_clean'] == 'own goal')
]

# Condition 2: When does the AWAY TEAM get a point?
# Case A: Away Team scores AND it is NOT an own goal
# Case B: Home Team scores AND it IS an own goal
away_point_conditions = [
    (df['team_side_clean'] == 'away') & (df['goal_type_clean'] != 'own goal'),
    (df['team_side_clean'] == 'home') & (df['goal_type_clean'] == 'own goal')
]

# --- APPLYING POINTS ---

# np.select(conditions, choices, default)
# If condition is met, give 1 point, otherwise 0
df['home_goal_flag'] = np.select(home_point_conditions, [1, 1], default=0)
df['away_goal_flag'] = np.select(away_point_conditions, [1, 1], default=0)

# --- CALCULATING SCORELINE (SAME AS BEFORE) ---

# Sort by Match and Minute first
# (Assuming you have already created 'minute_numeric' as discussed previously)
df = df.sort_values(by=['date', 'home_team', 'away_team', 'goal_minute_num'])

# Calculate Running Total
df['home_score_running'] = df.groupby(['date', 'home_team', 'away_team'])['home_goal_flag'].cumsum()
df['away_score_running'] = df.groupby(['date', 'home_team', 'away_team'])['away_goal_flag'].cumsum()

# Construct Scoreline
df['scoreline'] = df['home_score_running'].astype(str) + '-' + df['away_score_running'].astype(str)

# Verify with a quick look at Own Goals specifically
# print(df[df['goal_type_clean'] == 'own goal'][['home_team', 'away_team', 'team_side', 'goal_type', 'scoreline']].head())

In [11]:
df[df['match_id'] == 'UPL22/MBA/TOO/14-05']

,date,time,season,match_day,home_team,away_team,goal_minute,team_side,player_name,goal_type,...,match_id,round,match_half,goal_type_clean,team_side_clean,home_goal_flag,away_goal_flag,home_score_running,away_score_running,scoreline
1272,2022-05-14,4:00 pm,2021/22,29,Mbarara City FC,Tooro United FC,5,home,Jude Ssemugabi,Regular,...,UPL22/MBA/TOO/14-05,Round 2,First Half,regular,home,1,0,1,0,1-0
1273,2022-05-14,4:00 pm,2021/22,29,Mbarara City FC,Tooro United FC,12,home,Seiri Arigumaho,Regular,...,UPL22/MBA/TOO/14-05,Round 2,First Half,regular,home,1,0,2,0,2-0
1274,2022-05-14,4:00 pm,2021/22,29,Mbarara City FC,Tooro United FC,24,home,Jude Ssemugabi,Regular,...,UPL22/MBA/TOO/14-05,Round 2,First Half,regular,home,1,0,3,0,3-0
1275,2022-05-14,4:00 pm,2021/22,29,Mbarara City FC,Tooro United FC,31,away,Louis Mawa Bhoka,Regular,...,UPL22/MBA/TOO/14-05,Round 2,First Half,regular,away,0,1,3,1,3-1
1276,2022-05-14,4:00 pm,2021/22,29,Mbarara City FC,Tooro United FC,44,home,Jude Ssemugabi,Regular,...,UPL22/MBA/TOO/14-05,Round 2,First Half,regular,home,1,0,4,1,4-1
1277,2022-05-14,4:00 pm,2021/22,29,Mbarara City FC,Tooro United FC,55,home,Seiri Arigumaho,Regular,...,UPL22/MBA/TOO/14-05,Round 2,Second Half,regular,home,1,0,5,1,5-1
1278,2022-05-14,4:00 pm,2021/22,29,Mbarara City FC,Tooro United FC,69,home,Ronald Otti,Own Goal,...,UPL22/MBA/TOO/14-05,Round 2,Second Half,own goal,home,0,1,5,2,5-2
1279,2022-05-14,4:00 pm,2021/22,29,Mbarara City FC,Tooro United FC,76,home,Jude Ssemugabi,Regular,...,UPL22/MBA/TOO/14-05,Round 2,Second Half,regular,home,1,0,6,2,6-2
1280,2022-05-14,4:00 pm,2021/22,29,Mbarara City FC,Tooro United FC,83,home,Seiri Arigumaho,Regular,...,UPL22/MBA/TOO/14-05,Round 2,Second Half,regular,home,1,0,7,2,7-2
1281,2022-05-14,4:00 pm,2021/22,29,Mbarara City FC,Tooro United FC,88,home,Henry Kitegenyi,Regular,...,UPL22/MBA/TOO/14-05,Round 2,Second Half,regular,home,1,0,8,2,8-2


### Match Evolution and Decisive Impact Score

In [12]:
def calculate_match_evolution(df: pd.DataFrame) -> pd.DataFrame:
    """
    Reconstruct the scoreboard chronologically so we know the game state at the moment of every goal.

    Adds the following columns:
    - score_home_before: Home score before the goal
    - score_away_before: Away score before the goal
    - score_home_after: Home score after the goal
    - score_away_after: Away score after the goal
    - scoring_team_name: Name of the team that scored
    - is_home_goal: 1 if home team scored, 0 otherwise

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe with 'match_id', 'goal_minute_num', 'team_side', 'home_team', 'away_team'.

    Returns
    -------
    pd.DataFrame
        Dataframe with added score context columns.
    """
    df = df.copy()
    
    # Ensure logical order
    df = df.sort_values(["match_id", "goal_minute_num"]) 
    
    # Identify scoring side
    is_og = df["goal_type"] == 'Own Goal'

    df["is_home_goal"] = np.where(is_og,
    (df["team_side"] == 'away').astype(int), # Flip: Away OG = Home Goal
    (df["team_side"] == 'home').astype(int)  # Regular
    )
    
    df["is_away_goal"] = np.where(is_og,
    (df["team_side"] == 'home').astype(int), # Flip: Home OG = Away Goal
    (df["team_side"] == 'away').astype(int)  # Regular
    )
    
    # Determine scoring team name
    df["scoring_team_name"] = np.where(
        df["is_home_goal"] == 1, 
        df["home_team"], 
        df["away_team"]
    )
    
    # Calculate running scores using cumulative sum within each match
    df["score_home_after"] = df.groupby("match_id")["is_home_goal"].cumsum()
    df["score_away_after"] = df.groupby("match_id")["is_away_goal"].cumsum()
    
    # Calculate previous scores (shift and fill with 0)
    df["score_home_before"] = df.groupby("match_id")["score_home_after"].shift(1).fillna(0).astype(int)
    df["score_away_before"] = df.groupby("match_id")["score_away_after"].shift(1).fillna(0).astype(int)
    
    return df

In [13]:
df = calculate_match_evolution(df)

In [14]:
df.head()

,date,time,season,match_day,home_team,away_team,goal_minute,team_side,player_name,goal_type,...,home_score_running,away_score_running,scoreline,is_home_goal,is_away_goal,scoring_team_name,score_home_after,score_away_after,score_home_before,score_away_before
286,2019-09-11,4:00 pm,2019/20,4,BUL FC,Busoga United FC,72,away,Boban Bogere Zirintusa,Penalty,...,0,1,0-1,0,1,Busoga United FC,0,1,0,0
287,2019-09-11,4:00 pm,2019/20,4,BUL FC,Busoga United FC,80,away,Joel Madondo,Regular,...,0,2,0-2,0,1,Busoga United FC,0,2,0,1
288,2019-09-11,4:00 pm,2019/20,4,BUL FC,Busoga United FC,90+3,away,Lawrence Tezikya,Regular,...,0,3,0-3,0,1,Busoga United FC,0,3,0,2
203,2019-08-30,4:30 pm,2019/20,1,BUL FC,Express FC,13,home,Deogratious Ojok,Regular,...,1,0,1-0,1,0,BUL FC,1,0,0,0
215,2019-10-08,4:30 pm,2019/20,8,BUL FC,KCCA FC,34,home,Richard Wandyaka,Regular,...,1,0,1-0,1,0,BUL FC,1,0,0,0


In [15]:
def calculate_dis(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate Decisive Impact Score (DIS) for each goal.

    DIS = Time Weight * Outcome Weight

    Time Weights:
    - 0-15: 0.5
    - 16-30: 0.6
    - 31-45: 0.7
    - 46-60: 0.8
    - 61-75: 0.9
    - 76+: 1.0

    Outcome Weights:
    - Winning/Lead-taking Goal (Break Tie): 2.0
    - Equalizer (Losing -> Draw): 1.5
    - Cushion Goal (Lead -> Lead+1): 1.0
    - Consolation (Losing -> Losing less): 0.5
    
    Parameters
    ----------
    df : pd.DataFrame
        Dataframe containing match context (run calculate_match_evolution first).

    Returns
    -------
    pd.DataFrame
        Dataframe with 'dis_time_weight', 'dis_outcome_weight', and 'dis' columns.
    """
    df = df.copy()
    
    # 1. Calculate Time Weight
    def get_time_weight(minute):
        if pd.isna(minute): return 0
        if minute <= 15: return 0.5
        if minute <= 30: return 0.6
        if minute <= 45: return 0.7
        if minute <= 60: return 0.8
        if minute <= 75: return 0.9
        return 1.0
        
    df["dis_time_weight"] = df["goal_minute_num"].apply(get_time_weight)
    
    # 2. Calculate Outcome Weight
    # We need to know the state *before* the goal relative to the *scoring team*
    
    # Calculate score difference for the scoring team BEFORE the goal
    # If home scored: diff = home_before - away_before
    # If away scored: diff = away_before - home_before
    
    df["score_diff_before"] = np.where(
        df["is_home_goal"] == 1,
        df["score_home_before"] - df["score_away_before"],
        df["score_away_before"] - df["score_home_before"]
    )
    
    def get_outcome_weight(diff_before):
        if diff_before == 0:
            return 2.0  # Tie -> Lead (Potential Winner)
        elif diff_before == -1:
            return 1.5  # Losing by 1 -> Draw (Equalizer)
        elif diff_before > 0:
            return 1.0  # Winning -> Winning more (Cushion)
        else:
            return 0.5  # Losing by >1 -> Losing less (Consolation)
            
    df["dis_outcome_weight"] = df["score_diff_before"].apply(get_outcome_weight)
    
    # 3. Calculate Final DIS
    df["dis"] = df["dis_time_weight"] * df["dis_outcome_weight"]
    
    return df


In [16]:
df=calculate_dis(df)

In [17]:
df.head()

,date,time,season,match_day,home_team,away_team,goal_minute,team_side,player_name,goal_type,...,is_away_goal,scoring_team_name,score_home_after,score_away_after,score_home_before,score_away_before,dis_time_weight,score_diff_before,dis_outcome_weight,dis
286,2019-09-11,4:00 pm,2019/20,4,BUL FC,Busoga United FC,72,away,Boban Bogere Zirintusa,Penalty,...,1,Busoga United FC,0,1,0,0,0.9,0,2.0,1.8
287,2019-09-11,4:00 pm,2019/20,4,BUL FC,Busoga United FC,80,away,Joel Madondo,Regular,...,1,Busoga United FC,0,2,0,1,1.0,1,1.0,1.0
288,2019-09-11,4:00 pm,2019/20,4,BUL FC,Busoga United FC,90+3,away,Lawrence Tezikya,Regular,...,1,Busoga United FC,0,3,0,2,1.0,2,1.0,1.0
203,2019-08-30,4:30 pm,2019/20,1,BUL FC,Express FC,13,home,Deogratious Ojok,Regular,...,0,BUL FC,1,0,0,0,0.5,0,2.0,1.0
215,2019-10-08,4:30 pm,2019/20,8,BUL FC,KCCA FC,34,home,Richard Wandyaka,Regular,...,0,BUL FC,1,0,0,0,0.7,0,2.0,1.4


### Opponent Vulnerability Window

In [18]:
def calculate_ovw(df: pd.DataFrame, team_name: str) -> str:
    """
    Calculate Opponent Vulnerability Window (OVW).
    
    Identifies the 15-minute segment where a specific team *concedes* the most goals.

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe of all goals.
    team_name : str
        Name of the team to analyze (e.g. "Vipers SC").

    Returns
    -------
    str
        The description of the window (e.g., "76-90").
    """
    # Filter for goals conceded by the team
    # A team concedes if they are Home and Away scores, or they are Away and Home scores.
    # Logic: If I am Home, I concede if is_away_goal == 1.
    
    # First, identify the opponent for every row
    # If home_team == team_name, opponent scored if is_away_goal.
    # If away_team == team_name, opponent scored if is_home_goal.
    
    # Easier way: Filter rows where the *defending team* is team_name.
    # Defending Home Team concedes Away Goals.
    conceded_home = df[(df["home_team"] == team_name) & (df["is_away_goal"] == 1)]
    
    # Defending Away Team concedes Home Goals.
    conceded_away = df[(df["away_team"] == team_name) & (df["is_home_goal"] == 1)]
    
    conceded = pd.concat([conceded_home, conceded_away])
    
    if len(conceded) == 0:
        return "N/A"
        
    # Bin into 15 min windows
    def get_window(minute):
        if pd.isna(minute): return "Unknown"
        if minute <= 15: return "0-15"
        if minute <= 30: return "16-30"
        if minute <= 45: return "31-45"
        if minute <= 60: return "46-60"
        if minute <= 75: return "61-75"
        return "76-90+"
        
    window_counts = conceded["goal_minute_num"].apply(get_window).value_counts()
    
    if window_counts.empty:
        return "N/A"
        
    return window_counts.idxmax()

In [19]:
for season in df['season'].unique():
    season_df = df[df['season'] == season]
    ovw = calculate_ovw(season_df, "Mbarara City FC")
    print(f"Season {season}: OVW for Mbarara City FC = {ovw}")

Season 2019/20: OVW for Mbarara City FC = 76-90+
Season 2020/21: OVW for Mbarara City FC = 16-30
Season 2021/22: OVW for Mbarara City FC = 76-90+
Season 2022/23: OVW for Mbarara City FC = N/A
Season 2023/24: OVW for Mbarara City FC = 46-60
Season 2024/25: OVW for Mbarara City FC = 46-60


In [20]:
calculate_ovw(df, "Vipers SC")

'46-60'

In [21]:
for i in df['home_team'].unique().tolist():
    ovw = calculate_ovw(df, i)
    print(f"{i}: OVW = {ovw}")  

BUL FC: OVW = 76-90+
Busoga United FC: OVW = 16-30
Express FC: OVW = 76-90+
KCCA FC: OVW = 76-90+
Kyetume FC: OVW = 76-90+
Maroons FC: OVW = 76-90+
Mbarara City FC: OVW = 46-60
Onduparaka FC: OVW = 76-90+
Police FC: OVW = 46-60
Proline FC: OVW = 76-90+
SC Villa: OVW = 76-90+
Soltilo Bright Stars FC: OVW = 46-60
Tooro United FC: OVW = 76-90+
URA FC: OVW = 46-60
Vipers SC: OVW = 46-60
Wakiso Giants FC: OVW = 46-60
Kitara FC: OVW = 61-75
MYDA FC: OVW = 76-90+
UPDF FC: OVW = 61-75
Arua Hill SC: OVW = 76-90+
Entebbe UPPC FC: OVW = 76-90+
Blacks Power FC: OVW = 46-60
NEC FC: OVW = 16-30
Lugazi FC: OVW = 46-60
Mbale Heroes FC: OVW = 16-30


### Discipline Degradation Index (DDI)

In [22]:
def calculate_ddi(df: pd.DataFrame, team_name: str) -> float:
    """
    Calculate Discipline Degradation Index (DDI).

    Ratio of (Penalty + Own Goals) conceded in 2nd Half vs 1st Half.

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe of all goals.
    team_name : str
        Team to analyze.

    Returns
    -------
    float
        Ratio (2nd Half Bad Goals / 1st Half Bad Goals). 
        Returns infinity (or high number) if 0 in 1st half.
    """
    # Filter for bad goals conceded by team (when opponent scores)
    # Bad goals = Penalty or Own Goal
    bad_types = ["Penalty", "Own Goal"]
    
    # Conceded by home team = opponent (away) scored
    conceded_home = df[(df["home_team"] == team_name) & (df["team_side"] == "away")]
    # Conceded by away team = opponent (home) scored
    conceded_away = df[(df["away_team"] == team_name) & (df["team_side"] == "home")]
    conceded = pd.concat([conceded_home, conceded_away])
    
    bad_goals = conceded[conceded["goal_type"].isin(bad_types)]
    
    bad_h1 = len(bad_goals[bad_goals["match_half"] == "First Half"])
    bad_h2 = len(bad_goals[bad_goals["match_half"] == "Second Half"])
    
    if bad_h1 == 0:
        if bad_h2 > 0:
            return float('inf') # Infinite degradation
        else:
            return 0.0 # No bad goals at all
            
    return bad_h2 / bad_h1

In [23]:
# Calculate DDI for all teams across all seasons
ddi_results = []
for team in df['home_team'].unique().tolist():
    ddi = calculate_ddi(df, team)
    ddi_results.append({"Team": team, "DDI": ddi})

ddi_df = pd.DataFrame(ddi_results).sort_values(by="DDI", ascending=False)
ddi_df

,Team,DDI
21,Blacks Power FC,inf
23,Lugazi FC,inf
5,Maroons FC,3.000000
14,Vipers SC,2.500000
13,URA FC,2.285714
11,Soltilo Bright Stars FC,2.142857
4,Kyetume FC,1.666667
19,Arua Hill SC,1.500000
2,Express FC,1.500000
16,Kitara FC,1.500000
